# Taller 2 — Redes Neuronales Multicapa (MLP) y Backpropagation

**Asignatura:** Inteligencia Computacional Aplicada
**Docente:** Cesar Andrey Perdomo Charry
**Estudiante:** &lt;&lt;Nombre&gt;&gt;

---

Este notebook está listo para ejecutarse en **Google Colab**. Contiene la teoría
necesaria, referencias bibliográficas y el código base que debe completar
(bloques marcados con `# TODO`). No cambie los nombres ni los argumentos de
las funciones para que el resto del notebook siga funcionando.

## 1. Fundamento teórico: Perceptrón Multicapa (MLP) y Backpropagation

**¿Por qué una sola capa no basta?** En el Taller 1 vimos que el Perceptrón
Simple y Adaline solo pueden resolver problemas **linealmente separables**.
El problema XOR es el ejemplo clásico de un problema que *no* lo es, y por
eso requiere una arquitectura con al menos una **capa oculta**.

### Arquitectura del MLP

Una red multicapa se organiza en capas $k = 1,\dots,L$. Cada neurona $j$ de
la capa $k$ calcula:

$$net_j^{(k)} = \sum_i w_{ji}^{(k)} a_i^{(k-1)} + b_j^{(k)}, \qquad
a_j^{(k)} = f\left(net_j^{(k)}\right)$$

donde $a^{(0)} = x$ (la entrada) y $f$ es la función de activación de la capa.

### Algoritmo de Backpropagation

El error se mide típicamente como error cuadrático medio:

$$E = \frac{1}{2}\sum \left(d(x) - Y(x)\right)^2$$

El algoritmo ajusta los pesos por **descenso de gradiente**, propagando el
error desde la salida hacia las capas ocultas:

$$\delta^{(L)} = \left(d(x) - Y(x)\right) \odot f'\left(net^{(L)}\right)
\qquad \text{(capa de salida)}$$

$$\delta^{(k)} = \left(W^{(k+1)T}\delta^{(k+1)}\right) \odot
f'\left(net^{(k)}\right) \qquad \text{(capas ocultas)}$$

$$W^{(k)} \leftarrow W^{(k)} + \eta\, \delta^{(k)} \left(a^{(k-1)}\right)^T,
\qquad b^{(k)} \leftarrow b^{(k)} + \eta\, \delta^{(k)}$$

Opcionalmente se agrega un **término de momento** $\beta$:

$$\Delta W^{(k)} \leftarrow \eta\, \delta^{(k)} \left(a^{(k-1)}\right)^T +
\beta\, \Delta W^{(k)}_{\text{anterior}}$$

### Funciones de activación más comunes

| Función | $f(net)$ | $f'(net)$ | Comentario |
|---|---|---|---|
| Sigmoidal | $\dfrac{1}{1+e^{-net}}$ | $f(net)(1-f(net))$ | satura en los extremos (*vanishing gradient*) |
| Tangente hiperbólica | $\tanh(net)$ | $1-\tanh^2(net)$ | centrada en cero, converge algo más rápido |
| ReLU | $\max(0,net)$ | $1$ si $net>0$, $0$ si no | evita saturación, puede "apagar" neuronas |

### Referencias y lecturas complementarias

- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323(6088), 533–536. [doi.org/10.1038/323533a0](https://doi.org/10.1038/323533a0)
- Haykin, S. (2009). *Neural Networks and Learning Machines* (3rd ed.), Cap. 4. Pearson.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, Cap. 6. [deeplearningbook.org](https://www.deeplearningbook.org/) (gratis en línea)
- Nielsen, M. (2015). *Neural Networks and Deep Learning*, Cap. 2 (derivación paso a paso de backprop). [neuralnetworksanddeeplearning.com/chap2.html](http://neuralnetworksanddeeplearning.com/chap2.html) (libro gratuito)
- MathWorks. [Multilayer Shallow Neural Networks and Backpropagation Training](https://www.mathworks.com/help/deeplearning/ug/multilayer-shallow-neural-networks-and-backpropagation-training.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

## 2. Parámetros del modelo (totalmente parametrizable)

**Qué se espera en esta sección:** un diccionario `cfg` editable que controla
la arquitectura de la red y el entrenamiento. No requiere código adicional,
pero **sí** deberá modificar estos valores (número de neuronas ocultas,
función de activación, tasa de aprendizaje, etc.) en los distintos
numerales del taller.

**Dónde modificar:** cambie directamente los valores de `cfg` según el
experimento (XOR, Iris, Wine, Breast Cancer, etc.).

In [ ]:
cfg = {
    'n_inputs':      2,            # número de entradas
    'hidden_layers': [4],          # neuronas por capa oculta, ej. [4, 3]
    'n_outputs':     1,            # número de salidas
    'act_hidden':    'sigmoid',    # 'sigmoid' | 'tanh' | 'relu'
    'act_output':    'sigmoid',    # 'sigmoid' | 'tanh' | 'relu' | 'linear'
    'eta':           0.3,          # tasa de aprendizaje (eta)
    'momentum':      0.0,          # coeficiente de momento (beta), 0-1
    'max_epochs':    5000,
    'target_error':  1e-3,
}

## 3. Datos de entrenamiento

**Teoría breve:** el problema XOR asigna salida 1 cuando exactamente una de
las dos entradas es 1, y 0 en caso contrario. No existe una única línea
recta que separe las clases, por lo que un Perceptrón simple (Taller 1) *no
puede* resolverlo — de ahí la necesidad de la capa oculta.

**Qué se espera:** las matrices `X` (patrones, una fila por patrón) y `D`
(salidas deseadas) listas para entrenar.

**Dónde modificar:** reemplace `X` y `D` por el dataset correspondiente en
los numerales 4, 5 y 6 del taller (Iris, Wine, Breast Cancer Wisconsin,
`data_banknote_authentication.txt`).

In [ ]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)          # cada FILA es un patrón de entrada

D = np.array([[0], [1], [1], [0]], dtype=float)   # salida deseada d(x)

## 4. Funciones de activación

**Qué se espera:** `activation` ya está completa; usted debe completar `activation_deriv`, que se usa dentro de `backward_mlp` para calcular $f'(net)$.

**Dónde modificar:** complete el bloque `# TODO` de `activation_deriv`, usando la tabla de la Sección 1.

In [ ]:
def activation(net_in, tipo):
    if tipo == 'sigmoid':
        return 1.0 / (1.0 + np.exp(-net_in))
    elif tipo == 'tanh':
        return np.tanh(net_in)
    elif tipo == 'relu':
        return np.maximum(0.0, net_in)
    elif tipo == 'linear':
        return net_in
    else:
        raise ValueError(f'Función de activación no reconocida: {tipo}')


def activation_deriv(net_in, tipo):
    """
    Derivada de la función de activación, evaluada en net_in (el valor
    ANTES de activar). Se usa dentro de backward_mlp.

    TODO: complete cada caso (ver tabla de la Sección 1):
      'sigmoid' -> f(net)*(1-f(net))   [reutilice activation(net_in, 'sigmoid')]
      'tanh'    -> 1 - tanh(net_in)**2
      'relu'    -> 1 si net_in > 0, 0 si net_in <= 0
    """
    if tipo == 'sigmoid':
        dy = None
    elif tipo == 'tanh':
        dy = None
    elif tipo == 'relu':
        dy = None
    elif tipo == 'linear':
        dy = np.ones_like(net_in)
    else:
        raise ValueError(f'Función de activación no reconocida: {tipo}')
    return dy

## 5. Inicialización de la red

**Teoría:** los pesos se inicializan con valores aleatorios **pequeños** (no en cero) para romper la simetría entre neuronas de una misma capa; si todas partieran del mismo valor, aprenderían siempre lo mismo. Los sesgos (bias) usualmente se inicializan en cero.

**Qué se espera:** la función debe devolver un diccionario `net` con `net['W']`, `net['b']` (listas, una entrada por capa) y `net['dW_prev']`, `net['db_prev']` en ceros, para el término de momento.

**Dónde modificar:** complete el bloque `# TODO` dentro de `init_mlp`.

In [ ]:
def init_mlp(cfg):
    """
    Inicializa pesos y sesgos de todas las capas en el diccionario `net`.

    TODO:
      1) layer_sizes = [cfg['n_inputs']] + cfg['hidden_layers'] + [cfg['n_outputs']]
      2) Para k = 0 .. len(layer_sizes)-2, agregue a las listas:
         W_k = np.random.randn(layer_sizes[k+1], layer_sizes[k]) * 0.5
         b_k = np.zeros((layer_sizes[k+1], 1))
         dW_prev_k = np.zeros_like(W_k)
         db_prev_k = np.zeros_like(b_k)
    """
    net = {'W': [], 'b': [], 'dW_prev': [], 'db_prev': []}

    # --- complete aquí ---

    return net

## 6. Propagación hacia adelante (forward pass)

**Teoría:** dado un patrón de entrada, se calcula la salida de cada capa aplicando $a^{(k)} = f\left(W^{(k)}a^{(k-1)}+b^{(k)}\right)$ de manera secuencial hasta la capa de salida.

**Qué se espera:** la función debe devolver `y` (salida de la red) y `cache` con los valores intermedios (`cache['a']`, `cache['netv']`), necesarios para backpropagation.

**Dónde modificar:** complete el bloque `# TODO` dentro de `forward_mlp`.

In [ ]:
def forward_mlp(net, x, cfg):
    """
    x: vector columna, shape (n_inputs, 1)

    TODO:
      a = [x]
      netv = []
      L = len(net['W'])
      for k in range(L):
          net_k = net['W'][k] @ a[k] + net['b'][k]
          netv.append(net_k)
          if k == L - 1:
              a.append(activation(net_k, cfg['act_output']))
          else:
              a.append(activation(net_k, cfg['act_hidden']))
      y = a[-1]
      cache = {'a': a, 'netv': netv}
    """
    y = None
    cache = {'a': [], 'netv': []}

    # --- complete aquí ---

    return y, cache

## 7. Retropropagación del error (Backpropagation)

**Teoría:** implemente las fórmulas de la Sección 1 (deltas de salida y de las capas ocultas, y la actualización de pesos con momento opcional).

**Qué se espera:** la función debe actualizar y devolver `net` con los pesos ya ajustados para el patrón actual.

**Dónde modificar:** complete el bloque `# TODO` dentro de `backward_mlp`.

In [ ]:
def backward_mlp(net, cache, d, cfg):
    """
    Calcula los delta de cada capa (de salida hacia atrás) y actualiza los
    pesos:

        Capa de salida : delta_L = (d - y) * f'(net_L)
        Capas ocultas  : delta_k = (W_{k+1}^T @ delta_{k+1}) * f'(net_k)
        Actualización  : W_k += eta * delta_k @ a_{k-1}^T  (+ momentum)
                         b_k += eta * delta_k              (+ momentum)

    TODO: implemente el ciclo de backpropagation usando cache['a'] y
    cache['netv'].
    """

    # --- complete aquí ---

    return net

## 8. Entrenamiento (ciclo de épocas)

**Teoría:** en el *aprendizaje en línea* (online), los pesos se actualizan patrón por patrón. Se recorre el conjunto de entrenamiento en orden aleatorio en cada época para evitar sesgos de orden.

**Qué se espera:** al ejecutar esta celda, el error cuadrático medio (`error_hist`) debe decrecer época a época. Si no baja, revise las funciones completadas en las secciones 4-7.

**Dónde modificar:** no requiere cambios de código.

In [ ]:
net = init_mlp(cfg)
error_hist = []

for epoch in range(cfg['max_epochs']):
    epoch_error = 0.0
    idx = np.random.permutation(X.shape[0])

    for p in idx:
        x = X[p:p+1].T   # columna (n_inputs, 1)
        d = D[p:p+1].T   # columna (n_outputs, 1)

        y, cache = forward_mlp(net, x, cfg)
        net = backward_mlp(net, cache, d, cfg)

        epoch_error += 0.5 * np.sum((d - y) ** 2)

    epoch_error /= X.shape[0]
    error_hist.append(epoch_error)

    if epoch_error <= cfg['target_error']:
        print(f'Convergencia en la época {epoch} (error = {epoch_error:.6f})')
        break

## 9. Resultados y visualización

**Qué se espera:** una curva de error decreciente y, para XOR, salidas cercanas a 0 o 1 que coincidan con `d` para cada patrón. Compare con el Taller 1 (Perceptrón/Adaline), donde el modelo *no* podía converger en XOR.

**Dónde modificar:** no requiere cambios; reutilice esta celda para reportar resultados de cada dataset y configuración que pruebe.

In [ ]:
plt.figure()
plt.plot(error_hist, linewidth=1.4)
plt.xlabel('Época'); plt.ylabel('Error cuadrático medio')
plt.title('Curva de error de entrenamiento')
plt.grid(True)
plt.show()

print('\nResultados finales:')
for p in range(X.shape[0]):
    y, _ = forward_mlp(net, X[p:p+1].T, cfg)
    y_val = float(np.asarray(y).squeeze())
    print(f'x = {X[p]}  ->  y = {y_val:.4f}   (d = {D[p, 0]:g})')